# Step 3 — Model 2 DIN ablation

Train on sampled **train** slates, early-stop each variant on **val1 NDCG@10**, restore its best checkpoint, and compare variants on that same metric. No new negative sampling or row splitting. **Only train and val1 are loaded.** Val1 is a model-selection score, not an unbiased final test estimate.

Following the source DIN notebook, NDCG uses **purchase relevance** (`label == 3`), not graded gains: each search with at least one purchase contributes equally; zero-purchase searches are excluded. Ties are averaged over five seeded random tie-breaks. Training remains weighted binary cross entropy for any interaction, with weights `[1, 1, 5, 10]` for labels 0–3. Raw scores are not calibrated probabilities.

Inputs: `seq_train.parquet`, `seq_val1.parquet`, `vocab.json`, and `scalar_train.parquet` / `scalar_val1.parquet`. Scalar files may contain lists of structs or separate numeric lists. Slates are aligned by ID and candidate counts checked; candidate order within each slate must already match upstream.

Run top to bottom after adjusting paths. All outputs are generated when run on your data; this notebook contains no fabricated results.


## Setup

In [19]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set the paths in Settings to local folders.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from copy import deepcopy
from IPython.display import display

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU: training will be very slow")
print("numpy", np.__version__, "| pandas", pd.__version__, "| pyarrow", pa.__version__)

torch 2.11.0+cu130 | device: cuda | Tesla T4
numpy 2.4.6 | pandas 2.2.3 | pyarrow 23.0.1


## Settings
All variants share architecture, optimisation, and seeds. Start with one seed; add seeds to assess training variability.

In [22]:
# >>> Paths: change to where the Step 2b and Step 2c outputs live <<<
PROJECT = Path("/content/drive/MyDrive/Grp8Submission/pipeline_data")
SEQ_DIR = PROJECT / "step2_model2_seq"        # sequence inputs
SCALAR_DIR = PROJECT / "step2_model2_scalar"  # scalar features
OUT_DIR = PROJECT / "step3_model2"            # this notebook's outputs

SPLITS = ["train", "val1"]
TUNE_SPLIT = "val1"     # early stopping, tuning, ablation
for d in (SEQ_DIR, SCALAR_DIR, OUT_DIR.parent):
    assert d.is_dir(), f"{d} not found: check the shortcut to the shared Grp8Submission folder"
OUT_DIR.mkdir(exist_ok=True)

CFG = dict(
    # Inputs: switched off individually in Step 3 ablations
    channels=["click", "cart", "order"],
    use_query_terms=True,          # historical query terms
    use_name_terms=True,           # candidate product-name terms
    use_attributes=True,           # brand / L4 category / shop embeddings
    use_decay=True,                # explicit recency decay in attention
    use_scalars=True,              # Step 2c scalar features (Section 1.4.3)
    use_popularity=True,          # enabled by add_popularity

    # Fixed choices for Step 3
    use_cand_item_id=True,         # retain candidate item-ID embedding
    use_attention=True,            # retain DIN attention
    use_own_summaries=True,        # retain internal history summaries
    use_brand_cate_summaries=True, # included within Step 2c; no separate ablation
    use_dominant_brand_ids=True,   # exclude categorical brand IDs from numeric scalars

    # Architecture
    emb_dim=64,
    att_hidden=64,
    mlp_hidden=(256, 128),
    dropout=0.2,
    emb_dropout=0.0,

    # Optimisation
    lr=1e-3,
    weight_decay=0.0,
    emb_weight_decay=0.1,
    grad_clip=5.0,

    # Batching: keep whole searches together
    batch_cands=4096,
    eval_batch_cands=8192,

    # Early stopping on val1 purchase NDCG@10
    max_epochs=8,
    evals_per_epoch=2,
    patience=3,                    # validation checks without improvement

    # Runtime
    amp=True,                      # mixed precision when CUDA is available
    seed=0,
)
GRADED_WEIGHT = {0: 1.0, 1: 1.0, 2: 5.0, 3: 10.0}   # label -> sample weight (interim report, as Model 1)

SEQ_FIELDS = ["item", "brand_id", "cate_id_4", "shop_id", "action", "time_bucket", "has_query", "qry_terms", "days"]
ATTR_COLS = ["brand_id", "cate_id_4", "shop_id"]
SEEDS = [0,1,2]

## Step 1: Load the inputs

`load_split` is the reader from Step 2b: one parquet file per split, one row per search, turned back into per-channel grids `[searches, cap]` and flat candidate arrays (`row` = the candidate's search, `pos` = its position in the step 1 list).

In [23]:
VOCAB = json.loads((SEQ_DIR / "vocab.json").read_text())
CHANNELS = {ch: (v["actions"], v["window_days"]) for ch, v in VOCAB["channels"].items()}
HALF_LIFE = {ch: max(v["half_life_days"], 1e-3) for ch, v in VOCAB["channels"].items()}


def to_grid(arr):
    """Arrow (nested) fixed-size list array -> numpy array [n, k] or [n, k, m]."""
    shape, t = [len(arr)], arr.type
    while pa.types.is_fixed_size_list(t):
        shape.append(t.list_size)
        arr, t = arr.flatten(), t.value_type
    return arr.to_numpy(zero_copy_only=False).reshape(shape)


def load_split(name, out_dir=None):
    """(slate_id, user, seqs, cand) exactly as saved by Step 2b."""
    out_dir = out_dir or SEQ_DIR
    t = pq.read_table(out_dir / f"seq_{name}.parquet")
    user, seqs, cand = {}, {ch: {} for ch in CHANNELS}, {}
    for col in t.column_names:
        arr = t.column(col).combine_chunks()
        prefix, _, field = col.partition("_")
        if prefix in CHANNELS:
            seqs[prefix][field] = to_grid(arr) if pa.types.is_fixed_size_list(arr.type) \
                                  else arr.to_numpy(zero_copy_only=False)
        elif prefix == "cand":
            values = arr.flatten()
            cand[field] = to_grid(values) if pa.types.is_fixed_size_list(values.type) else values.to_numpy()
            if "row" not in cand:
                counts = arr.value_lengths().to_numpy()
                cand["row"] = np.repeat(np.arange(len(arr)), counts).astype(np.int32)
                cand["pos"] = (np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)).astype(np.int16)
        elif col != "slate_id":
            user[col] = arr.to_numpy()
    return t.column("slate_id").to_numpy(), user, seqs, cand


RAW = {split: load_split(split) for split in SPLITS}
for split, (sid, user, seqs, cand) in RAW.items():
    caps = {ch: s["item"].shape[1] for ch, s in seqs.items()}
    print(f"{split}: {len(sid):,} searches | {len(cand['item']):,} candidates | "
          f"purchases {int((cand['label'] == 3).sum()):,} | channel caps {caps}")

train: 122,083 searches | 2,608,369 candidates | purchases 14,774 | channel caps {'click': 65, 'cart': 80, 'order': 49}
val1: 17,799 searches | 1,589,673 candidates | purchases 2,016 | channel caps {'click': 65, 'cart': 80, 'order': 49}


## Scalar inputs
The recursive loader expands each candidate struct into named numeric columns. Train-only scaling and NaN indicators are retained. Brand/category affinity ratios and history counts are included automatically.

Dominant-brand IDs are categorical identifiers, so they are **excluded by default** from the numeric MLP inputs (`use_dominant_brand_ids=False`); product brand embeddings remain enabled. This avoids imposing a numeric ordering on brand IDs. The scalar manifest records exactly which inputs each variant uses.

Removing a channel removes its sequence and channel-prefixed scalar features. Full-history aggregate summaries remain: channel ablations measure the channel-specific contribution conditional on those aggregates.


In [24]:
def read_step2c(split, slate_id, cand_counts):
    """Load candidate scalar features, aligned to sequence slate/candidate order."""
    path = SCALAR_DIR / f"scalar_{split}.parquet"

    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found: set SCALAR_DIR, "
            "or CFG['use_scalars'] = False to train without it"
        )

    t = pq.read_table(path)
    sid = t.column("slate_id").to_numpy()
    pos = pd.Series(np.arange(len(sid)), index=sid)

    if pos.index.duplicated().any():
        raise ValueError(f"{split}: duplicate slate_id in Step 2c file")

    missing = np.setdiff1d(slate_id, sid)
    if len(missing):
        raise ValueError(
            f"{split}: {len(missing):,} searches missing from Step 2c file"
        )

    order = pa.array(pos.loc[slate_id].to_numpy(), type=pa.int64())
    out = {}

    def extract(name, values):
        # A candidate struct contains one field per scalar feature.
        if pa.types.is_struct(values.type):
            for field in values.type:
                child_name = f"{name}__{field.name}" if name else field.name
                # Propagates null parent structs to their child values.
                child = pc.struct_field(values, field.name)
                extract(child_name, child)
            return

        if not (
            pa.types.is_integer(values.type)
            or pa.types.is_floating(values.type)
            or pa.types.is_boolean(values.type)
            or pa.types.is_decimal(values.type)
            or pa.types.is_null(values.type)
        ):
            raise TypeError(
                f"{split}: feature '{name}' has unsupported type "
                f"{values.type}; expected a numeric scalar"
            )

        if name in out:
            raise ValueError(f"{split}: duplicate feature name '{name}'")

        out[name] = (
            pc.cast(values, pa.float64())
            .to_numpy(zero_copy_only=False)
            .astype(np.float64)
        )

    for col in t.column_names:
        if col == "slate_id":
            continue

        arr = pc.take(t.column(col).combine_chunks(), order)

        if not (
            pa.types.is_list(arr.type)
            or pa.types.is_large_list(arr.type)
            or pa.types.is_fixed_size_list(arr.type)
        ):
            raise TypeError(
                f"{split}: column '{col}' has type {arr.type}; "
                "expected one candidate list per slate"
            )

        if arr.null_count:
            raise ValueError(
                f"{split}: column '{col}' contains null candidate lists"
            )

        lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)
        if not np.array_equal(lens, cand_counts):
            raise ValueError(
                f"{split}: '{col}' has a different number of "
                "candidates than the sequence inputs"
            )

        values = pc.list_flatten(arr)

        if pa.types.is_struct(values.type):
            # Keep the original feature names for channel filtering.
            extract("", values)
        else:
            extract(col, values)

    return out


def item_popularity():
    """Times each known item appears in TRAIN histories, per channel (train-only statistic).
    Unknown (1) and padding (0) get 0. Counts come from the capped channel grids."""
    pop = {}
    for ch, s in RAW["train"][2].items():
        c = np.bincount(s["item"].ravel(), minlength=VOCAB["n_items"]).astype(np.float64)
        c[:2] = 0.0
        pop[ch] = c
    return pop


POPULARITY = item_popularity()


def own_summaries(user, seqs, cand):
    """Per-candidate copies of the Step 2b per-channel summaries."""
    row = cand["row"]
    # raw counts and days: ScalarTransform applies log1p once (they are outside [0, 1])
    out = {"history_len": user["history_len"].astype(np.float64)[row],
           "cand_has_metadata": cand["has_metadata"].astype(np.float64)}
    for ch, s in seqs.items():
        out[f"{ch}_count"] = s["full_len"].astype(np.float64)[row]
        out[f"{ch}_days_since_last"] = s["days_since_last"].astype(np.float64)[row]   # NaN if the channel is empty
        out[f"{ch}_empty"] = (s["full_len"] == 0).astype(np.float64)[row]
        out[f"{ch}_popularity"] = POPULARITY[ch][cand["item"]]                    # candidate popularity (raw count)
    return out


def raw_scalars(split):
    sid, user, seqs, cand = RAW[split]
    feats = {f"own__{k}": v for k, v in own_summaries(user, seqs, cand).items()}
    counts = np.bincount(cand["row"], minlength=len(sid))
    if CFG["use_scalars"] or SCALAR_DIR.exists():
        try:
            feats.update({f"step2c__{k}": v for k, v in read_step2c(split, sid, counts).items()})
        except FileNotFoundError as e:
            if CFG["use_scalars"]:
                raise
            print(e)
    return feats


class ScalarTransform:
    """Fitted on train: signed log1p for features outside [0, 1], standardise, NaN -> 0 plus a flag."""
    def fit(self, feats):
        self.names, self.log, self.mean, self.std, self.flag = [], {}, {}, {}, []
        for k, v in feats.items():
            finite = v[np.isfinite(v)]
            self.log[k] = bool(len(finite)) and (finite.min() < 0 or finite.max() > 1)
            x = self._log(k, v)
            xf = x[np.isfinite(x)]
            self.mean[k] = float(xf.mean()) if len(xf) else 0.0
            sd = float(xf.std()) if len(xf) else 0.0
            self.std[k] = sd if sd > 1e-12 else 1.0
            self.names.append(k)
            if np.isnan(v).any():
                self.flag.append(k)
        self.columns = self.names + [f"{k}__missing" for k in self.flag]
        return self

    def _log(self, k, v):
        return np.sign(v) * np.log1p(np.abs(v)) if self.log[k] else v

    def transform(self, feats):
        if set(feats) != set(self.names):
            raise ValueError(f"scalar features differ from train: {set(feats) ^ set(self.names)}")
        cols = []
        for k in self.names:
            x = (self._log(k, feats[k]) - self.mean[k]) / self.std[k]
            cols.append(np.where(np.isfinite(x), x, 0.0))
        cols += [np.isnan(feats[k]).astype(np.float64) for k in self.flag]
        return np.column_stack(cols).astype(np.float32)


raw_sc = {split: raw_scalars(split) for split in RAW}
SCALER = ScalarTransform().fit(raw_sc["train"])
SCALARS = {split: SCALER.transform(f) for split, f in raw_sc.items()}
del raw_sc
SCALAR_NAMES = SCALER.columns
n_step2c = sum(c.startswith("step2c__") and not c.endswith("__missing") for c in SCALAR_NAMES)
print(
    f"{len(SCALAR_NAMES)} scalar inputs: "
    f"{n_step2c} from Step 2c, "
    f"{sum(c.startswith('own__') and not c.endswith('__missing') for c in SCALAR_NAMES)} internal summaries, "
    f"{len(SCALER.flag)} missing-value flags\n"
)

for i, name in enumerate(SCALAR_NAMES, start=1):
    print(f"{i:2d}. {name}")


def scalar_columns(cfg):
    """Select scalar columns according to the ablation configuration."""
    dropped = set(CHANNELS) - set(cfg["channels"])
    keep = []

    for i, name in enumerate(SCALAR_NAMES):
        # Apply the same selection rules to a feature and its missingness flag.
        base = (
            name[:-len("__missing")]
            if name.endswith("__missing")
            else name
        )
        src, _, feat = base.partition("__")

        # All Step 2c scalar features.
        if src == "step2c" and not cfg["use_scalars"]:
            continue

        # Internally assembled sequence summaries, including popularity.
        if src == "own" and not cfg.get("use_own_summaries", True):
            continue

        # Dominant-brand IDs from any source.
        if (
            "dominant_brand" in feat
            and not cfg.get("use_dominant_brand_ids", False)
        ):
            continue

        # Step 2c brand/category summaries.
        # History-count features remain because their names lack these tokens.
        if (
            src == "step2c"
            and not cfg.get("use_brand_cate_summaries", True)
            and any(token in feat for token in ("brand", "cate"))
        ):
            continue

        # Candidate popularity.
        if (
            feat.endswith("_popularity")
            and not cfg.get("use_popularity", False)
        ):
            continue

        # Channel-specific scalars.
        if any(
            feat.startswith(f"{ch}_")
            or feat.startswith(f"missing_value_flags_{ch}_")
            for ch in dropped
        ):
            continue

        keep.append(i)

    return np.asarray(keep, dtype=np.int64)

45 scalar inputs: 16 from Step 2c, 14 internal summaries, 15 missing-value flags

 1. own__history_len
 2. own__cand_has_metadata
 3. own__click_count
 4. own__click_days_since_last
 5. own__click_empty
 6. own__click_popularity
 7. own__cart_count
 8. own__cart_days_since_last
 9. own__cart_empty
10. own__cart_popularity
11. own__order_count
12. own__order_days_since_last
13. own__order_empty
14. own__order_popularity
15. step2c__hist_dominant_brand
16. step2c__brand_affinity_ratio
17. step2c__cate_affinity_ratio
18. step2c__hist_item_count
19. step2c__click_hist_dominant_brand
20. step2c__click_brand_affinity_ratio
21. step2c__click_cate_affinity_ratio
22. step2c__click_hist_item_count
23. step2c__cart_hist_dominant_brand
24. step2c__cart_brand_affinity_ratio
25. step2c__cart_cate_affinity_ratio
26. step2c__cart_hist_item_count
27. step2c__order_hist_dominant_brand
28. step2c__order_brand_affinity_ratio
29. step2c__order_cate_affinity_ratio
30. step2c__order_hist_item_count
31. own__

## Step 3: Batching

A batch is a set of whole searches: each search's histories are encoded once and shared by all of its candidates (`row` points each candidate to its search). Training batches are filled up to `batch_cands` candidates.

In [25]:
class SplitData:
    def __init__(self, split):
        self.split = split
        self.slate_id, self.user, self.seqs, self.cand = RAW[split]
        self.scalars = SCALARS[split]
        n = len(self.slate_id)
        self.counts = np.bincount(self.cand["row"], minlength=n)
        self.starts = np.concatenate([[0], np.cumsum(self.counts)[:-1]])
        self.has_purchase = np.bincount(self.cand["row"], weights=(self.cand["label"] == 3), minlength=n) > 0
        assert np.all(np.diff(self.cand["row"]) >= 0), "candidates must be grouped by search"

    def batches(self, max_cands, shuffle=False, rng=None, slates=None):
        idx = np.arange(len(self.slate_id)) if slates is None else np.asarray(slates)
        if shuffle:
            idx = rng.permutation(idx)
        out, cur, n = [], [], 0
        for s in idx:
            c = self.counts[s]
            if cur and n + c > max_cands:
                out.append(np.array(cur)); cur, n = [], 0
            cur.append(s); n += c
        if cur:
            out.append(np.array(cur))
        return out

    def gather(self, sl, channels, scalar_cols):
        """Numpy arrays for one batch of searches `sl`."""
        lens = self.counts[sl]
        cidx = np.repeat(self.starts[sl], lens) + (np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens))
        return {
            "seq": {ch: {f: self.seqs[ch][f][sl] for f in SEQ_FIELDS} for ch in channels},
            "cand": {f: self.cand[f][cidx] for f in ["item", *ATTR_COLS, "name_terms", "label"]},
            "row": np.repeat(np.arange(len(sl)), lens),
            "scalars": self.scalars[cidx][:, scalar_cols],
            "cidx": cidx,
        }


DATA = {split: SplitData(split) for split in RAW}
for split, d in DATA.items():
    print(f"{split}: {d.has_purchase.sum():,} searches with a purchase "
          f"(of {len(d.slate_id):,}); median {int(np.median(d.counts))} candidates per search")

train: 14,408 searches with a purchase (of 122,083); median 21 candidates per search
val1: 1,945 searches with a purchase (of 17,799); median 90 candidates per search


In [26]:
def to_device(b):
    def t(x, float_=False):
        x = torch.from_numpy(np.ascontiguousarray(x))
        return (x.float() if float_ else x.long()).to(DEVICE, non_blocking=True)
    return {
        "seq": {ch: {f: t(v, float_=(f == "days")) for f, v in s.items()} for ch, s in b["seq"].items()},
        "cand": {f: t(v) for f, v in b["cand"].items()},
        "row": t(b["row"]),
        "scalars": t(b["scalars"], float_=True),
    }

## DIN network
Shared product/text embeddings → candidate-conditioned attention per channel → concatenate channel interests, candidate representation and scalars → MLP score. A masked mean-pooling ablation replaces learned attention and recency weighting together. Removing query text also removes its availability flag. Removing decay retains the learned time-bucket embedding.


In [27]:
class Model2(nn.Module):
    def __init__(self, cfg, n_scalar):
        super().__init__()
        d, h = cfg["emb_dim"], cfg["att_hidden"]
        self.cfg = cfg
        self.item = nn.Embedding(VOCAB["n_items"], d, padding_idx=0)
        self.attr = nn.ModuleDict({c: nn.Embedding(VOCAB[f"n_{c}"], d, padding_idx=0) for c in ATTR_COLS})
        self.term = nn.Embedding(VOCAB["n_terms"], d, padding_idx=0)
        self.action = nn.Embedding(VOCAB["n_actions"], d, padding_idx=0)
        self.tbucket = nn.Embedding(VOCAB["n_time_buckets"], d, padding_idx=0)
        self.emb_drop = nn.Dropout(cfg.get("emb_dropout", 0.0))
        self.event = nn.Linear(4 * d + 1, d)
        self.cand = nn.Linear(2 * d, d)
        self.att = nn.ModuleDict({ch: nn.Sequential(nn.Linear(4 * d, h), nn.ReLU(), nn.Linear(h, 1))
                                  for ch in cfg["channels"]})
        layers, width = [], len(cfg["channels"]) * d + d + n_scalar
        for out in cfg["mlp_hidden"]:
            layers += [nn.Linear(width, out), nn.ReLU(), nn.Dropout(cfg["dropout"])]
            width = out
        self.mlp = nn.Sequential(*layers, nn.Linear(width, 1))
        self.embedding_tables = [self.item, *self.attr.values(), self.term]
        for emb in [self.item, *self.attr.values(), self.term, self.action, self.tbucket]:
            nn.init.normal_(emb.weight, std=0.05)
            with torch.no_grad():
                emb.weight[0].zero_()

    def product(self, x, use_item=True):
        v = self.item(x["item"]) if use_item else 0.0
        if self.cfg["use_attributes"]:
            for c in ATTR_COLS:
                v = v + self.attr[c](x[c])
        if not torch.is_tensor(v):          # neither item ID nor attributes: a zero vector
            v = torch.zeros(*x["item"].shape, self.item.embedding_dim, device=x["item"].device)
        return self.emb_drop(v)

    def words(self, terms):
        e = self.emb_drop(self.term(terms))
        m = (terms > 0).unsqueeze(-1).to(e.dtype)
        return (e * m).sum(-2) / m.sum(-2).clamp(min=1.0)

    def forward(self, b, return_attn=False):
        c = b["cand"]
        name = self.words(c["name_terms"]) if self.cfg["use_name_terms"] else \
               torch.zeros(len(c["item"]), self.item.embedding_dim, device=c["item"].device)
        cvec = self.cand(torch.cat([self.product(c, use_item=self.cfg.get("use_cand_item_id", True)), name], -1))  # [N, d]
        row, parts, attn = b["row"], [], {}
        for ch in self.cfg["channels"]:
            s = b["seq"][ch]
            if self.cfg["use_query_terms"]:
                q = self.words(s["qry_terms"])
            else:
                q = torch.zeros(*s["item"].shape, self.item.embedding_dim, device=s["item"].device)
            ev = self.event(torch.cat([self.product(s), self.action(s["action"]), self.tbucket(s["time_bucket"]),
                                       q, (s["has_query"] if self.cfg["use_query_terms"] else torch.zeros_like(s["has_query"])).unsqueeze(-1).float()], -1))  # [B, L, d]
            H = ev[row]                                                               # [N, L, d]
            C = cvec.unsqueeze(1).expand_as(H)
            logits = self.att[ch](torch.cat([H, C, H - C, H * C], -1)).squeeze(-1).float()
            if self.cfg["use_decay"]:
                logits = logits - math.log(2) * s["days"][row] / HALF_LIFE[ch]
            mask = s["action"][row] > 0
            w = torch.softmax(logits.masked_fill(~mask, -1e9), dim=-1) * mask.float()  # empty channel -> all 0
            if not self.cfg.get("use_attention", True):
                w = mask.float() / mask.sum(-1, keepdim=True).clamp(min=1)
            parts.append((w.unsqueeze(-1) * H.float()).sum(1))
            if return_attn:
                attn[ch] = w.detach()
        x = torch.cat(parts + [cvec.float(), b["scalars"]], -1)
        out = self.mlp(x).squeeze(-1)
        return (out, attn) if return_attn else out


WEIGHT_T = torch.tensor([GRADED_WEIGHT[k] for k in range(4)], device=DEVICE)


def weighted_bce(logit, label):
    """Graded objective: target = any interaction, weight = 1 / 1 / 5 / 10 for labels 0 / 1 / 2 / 3."""
    w = WEIGHT_T[label]
    y = (label >= 1).float()
    loss = (F.binary_cross_entropy_with_logits(logit.float(), y, reduction="none") * w).sum() / w.sum()
    return loss, float(w.sum())

## Step 5: Metrics

`per_slate_metrics` and `paired_diff` are copied unchanged from the Model 1 / LightGBM ablation notebooks, so every model in the report is scored identically: purchase relevance (label 3), per-search NDCG@10, Hit@10, Hit@1 and MRR, ties broken randomly and averaged over 5 tie-breaks, searches without a purchase excluded.

In [28]:
METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True,
                      n_tiebreaks=5, seed=0):
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)
    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))
        g = gain[order]
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first
    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def paired_diff(a, b, col="ndcg@10", n_boot=2000, seed=0):
    # mean of a - b over slates, with a 95% bootstrap interval (a and b: per-slate tables on the same slates)
    assert (a["slate_id"].to_numpy() == b["slate_id"].to_numpy()).all(), "slates not aligned"
    d = a[col].to_numpy() - b[col].to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0,len(d),size=(min(100,n_boot-i),len(d)))].mean(axis=1)
                           for i in range(0,n_boot,100)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"diff": d.mean(), "ci_low": lo, "ci_high": hi}


def candidate_frame(data, scores, slates=None):
    """slate_id / position / label / score for the candidates of `slates` (default: all)."""
    rows = np.arange(len(data.slate_id)) if slates is None else np.asarray(slates)
    lens = data.counts[rows]
    cidx = np.repeat(data.starts[rows], lens) + (np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens))
    return pd.DataFrame({"slate_id": data.slate_id[data.cand["row"][cidx]], "position": data.cand["pos"][cidx],
                         "label": data.cand["label"][cidx], "score": scores[cidx]})

In [29]:
@torch.no_grad()
def predict(model, data, slates=None):
    """Raw logits for every candidate of `slates` (others NaN), in the data's candidate order."""
    model.eval()
    cols = scalar_columns(model.cfg)
    scores = np.full(len(data.cand["item"]), np.nan, dtype=np.float64)
    for sl in data.batches(model.cfg["eval_batch_cands"], slates=slates):
        b = data.gather(sl, model.cfg["channels"], cols)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=model.cfg["amp"] and DEVICE.type == "cuda"):
            out = model(to_device(b))
        scores[b["cidx"]] = out.float().cpu().numpy()
    return scores


def evaluate(model, data, slates=None):
    """Per-search metrics on searches with a purchase."""
    slates = np.flatnonzero(data.has_purchase) if slates is None else slates
    return per_slate_metrics(candidate_frame(data, predict(model, data, slates), slates), "score")

## Training
Each run starts from fresh weights and its assigned seed. The maximum budget and patience are identical; every variant restores its own highest val1 NDCG@10 checkpoint.

In [30]:
def make_optimizer(model, cfg):
    """AdamW. Embedding tables (items, brands, categories, shops, words) get `emb_weight_decay`;
    everything else gets `weight_decay`. With both at 0 this is plain Adam."""
    emb_ids = {id(p) for t in model.embedding_tables for p in t.parameters()}
    emb = [p for p in model.parameters() if id(p) in emb_ids]
    rest = [p for p in model.parameters() if id(p) not in emb_ids]
    return torch.optim.AdamW([{"params": emb, "weight_decay": cfg.get("emb_weight_decay", 0.0)},
                              {"params": rest, "weight_decay": cfg["weight_decay"]}], lr=cfg["lr"])


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu():
    """Release GPU memory from models that are no longer referenced."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()




In [31]:
def make_optimizer(model, cfg):
    """AdamW. Embedding tables (items, brands, categories, shops, words) get `emb_weight_decay`;
    everything else gets `weight_decay`. With both at 0 this is plain Adam."""
    emb_ids = {id(p) for t in model.embedding_tables for p in t.parameters()}
    emb = [p for p in model.parameters() if id(p) in emb_ids]
    rest = [p for p in model.parameters() if id(p) not in emb_ids]
    return torch.optim.AdamW([{"params": emb, "weight_decay": cfg.get("emb_weight_decay", 0.0)},
                              {"params": rest, "weight_decay": cfg["weight_decay"]}], lr=cfg["lr"])


def train_model(cfg, fixed_points=None, verbose=True):
    """Train one Model 2, checking val1 `evals_per_epoch` times per epoch.
    fixed_points=None: early stopping on val1 NDCG@10 (keeps the best check's weights);
    otherwise train exactly that many check intervals (used by the ablation)."""
    set_seed(cfg["seed"])
    cols = scalar_columns(cfg)
    model = Model2(cfg, len(cols)).to(DEVICE)
    opt = make_optimizer(model, cfg)
    use_amp = cfg["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = np.random.default_rng(cfg["seed"])
    tr, va = DATA["train"], DATA[TUNE_SPLIT]
    k = max(1, int(cfg.get("evals_per_epoch", 1)))
    history, best, best_state, best_point, bad, point = [], -1.0, None, 0, 0, 0
    t0, total, wsum = time.time(), 0.0, 0.0
    for epoch in range(1, cfg["max_epochs"] + 1):
        batches = tr.batches(cfg["batch_cands"], shuffle=True, rng=rng)
        parts = np.array_split(np.arange(len(batches)), k)
        for part in parts:
            if len(part) == 0:
                continue
            model.train()
            for bi in part:
                b = to_device(tr.gather(batches[bi], cfg["channels"], cols))
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    out = model(b)
                loss, w = weighted_bce(out, b["cand"]["label"])
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(opt)
                scaler.update()
                total += loss.item() * w; wsum += w
            point += 1
            val = evaluate(model, va)
            rec = {"check": point, "epoch": round(point / k, 2), "train_loss": total / max(wsum, 1e-12),
                   **{f"{TUNE_SPLIT}_{m}": val[m].mean() for m in METRICS}, "minutes": (time.time() - t0) / 60}
            history.append(rec)
            t0, total, wsum = time.time(), 0.0, 0.0
            if verbose:
                print(f"epoch {rec['epoch']:.2f}: train loss {rec['train_loss']:.4f} | "
                      f"{TUNE_SPLIT} NDCG@10 {rec[TUNE_SPLIT + '_ndcg@10']:.4f} | "
                      f"{TUNE_SPLIT} Hit@10 {rec[TUNE_SPLIT + '_hit@10']:.4f} | {rec['minutes']:.1f} min")
            if fixed_points is not None:
                if point >= fixed_points:
                    return model, pd.DataFrame(history), point
                continue
            if rec[f"{TUNE_SPLIT}_ndcg@10"] > best:
                best, best_point, bad = rec[f"{TUNE_SPLIT}_ndcg@10"], point, 0
                best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    model.load_state_dict(best_state)
                    return model, pd.DataFrame(history), best_point
    if fixed_points is None:
        model.load_state_dict(best_state)
        return model, pd.DataFrame(history), best_point
    return model, pd.DataFrame(history), point




## Ablation runs and saved outputs
One input group changes from the full reference in each removal experiment. `no_step2c_scalars` retains internally computed summaries; `no_all_scalars` removes both. The brand/category summary removal retains history counts and all product embeddings. Popularity is a separate addition experiment.

The table reports the selected checkpoint's val1 NDCG@10, difference from the full reference at the same seed, training time and input count. Checkpoints, histories, per-slate metrics, and a feature manifest are saved per run. These experiments can take substantial time and GPU memory; reduce batch candidate budgets if needed (a single slate is never split).


In [32]:
ABLATIONS = {
    "full": {},
    "no_click": {"channels": ["cart", "order"]},
    "no_cart": {"channels": ["click", "order"]},
    "no_order": {"channels": ["click", "cart"]},
    "no_history_query": {"use_query_terms": False},
    "no_candidate_name": {"use_name_terms": False},
    "no_attributes": {"use_attributes": False},
    "no_recency_decay": {"use_decay": False},
    "no_step2c_scalars": {"use_scalars": False},
    "no_popularity": {"use_popularity": False},
    "no_candidate_item_ID":  {"use_cand_item_id": False},
    "no_attention":  {"use_attention": False},
    "no_seq_summaries":  {"use_own_summaries": False},
    "no_scalar_summaries":  {"use_brand_cate_summaries": False},
    "no_dominant_brands":  {"use_dominant_brand_ids": False}
}

In [33]:
for name, overrides in ABLATIONS.items():
    cfg = {**CFG, **overrides}
    cols = scalar_columns(cfg)

    print(f"\n{name}: {len(cols)} scalar inputs")
    for i in cols:
        print(f"  {SCALAR_NAMES[i]}")


full: 45 scalar inputs
  own__history_len
  own__cand_has_metadata
  own__click_count
  own__click_days_since_last
  own__click_empty
  own__click_popularity
  own__cart_count
  own__cart_days_since_last
  own__cart_empty
  own__cart_popularity
  own__order_count
  own__order_days_since_last
  own__order_empty
  own__order_popularity
  step2c__hist_dominant_brand
  step2c__brand_affinity_ratio
  step2c__cate_affinity_ratio
  step2c__hist_item_count
  step2c__click_hist_dominant_brand
  step2c__click_brand_affinity_ratio
  step2c__click_cate_affinity_ratio
  step2c__click_hist_item_count
  step2c__cart_hist_dominant_brand
  step2c__cart_brand_affinity_ratio
  step2c__cart_cate_affinity_ratio
  step2c__cart_hist_item_count
  step2c__order_hist_dominant_brand
  step2c__order_brand_affinity_ratio
  step2c__order_cate_affinity_ratio
  step2c__order_hist_item_count
  own__click_days_since_last__missing
  own__cart_days_since_last__missing
  own__order_days_since_last__missing
  step2c__hist

In [34]:
RUN_VARIANTS = list(ABLATIONS)
assert "full" in RUN_VARIANTS
assert SEEDS and len(SEEDS) == len(set(SEEDS))

runs_path = OUT_DIR / "ablation_runs.csv"
manifest_path = OUT_DIR / "scalar_manifest.json"

previous_rows = {}
if runs_path.exists():
    saved = pd.read_csv(runs_path)
    previous_rows = {
        (r["variant"], int(r["seed"])): r
        for r in saved.to_dict("records")
    }

manifest = (
    json.loads(manifest_path.read_text())
    if manifest_path.exists()
    else {}
)
rows = []


def json_normalise(value):
    # Makes tuples and lists comparable after saving/loading JSON.
    return json.loads(json.dumps(value))


for seed in SEEDS:
    reference = None

    for name in ["full"] + [n for n in RUN_VARIANTS if n != "full"]:
        cfg = deepcopy(CFG)
        cfg.update(deepcopy(ABLATIONS[name]))
        cfg["seed"] = seed

        tag = f"{name}_seed{seed}"
        run_dir = OUT_DIR / tag
        cols = scalar_columns(cfg)
        scalar_names = [SCALAR_NAMES[i] for i in cols]

        metrics_path = run_dir / "val1_per_slate.parquet"
        config_path = run_dir / "config.json"
        result_path = run_dir / "result.json"

        # result.json supports resuming even if the aggregate CSV is missing.
        cached_row = (
            json.loads(result_path.read_text())
            if result_path.exists()
            else previous_rows.get((name, seed))
        )

        complete = all(
            (run_dir / filename).exists()
            for filename in (
                "history.csv",
                "val1_per_slate.parquet",
                "checkpoint.pt",
                "config.json",
            )
        )

        cached_names = (
            cached_row.get("scalar_names", manifest.get(tag))
            if cached_row is not None
            else None
        )

        reuse = (
            complete
            and cached_row is not None
            and json.loads(config_path.read_text()) == json_normalise(cfg)
            and cached_names == scalar_names
        )

        print(f"\n{tag}: {len(cols)} scalar inputs")

        if reuse:
            print("Loading saved results.")
            ps = (
                pd.read_parquet(metrics_path)
                .sort_values("slate_id")
                .reset_index(drop=True)
            )
            best_point = int(cached_row["best_check"])
            minutes = float(cached_row["minutes"])

        else:
            print("No matching completed run found; training.")
            run_dir.mkdir(parents=True, exist_ok=True)

            # Remove completion metadata before replacing an incomplete/stale run.
            result_path.unlink(missing_ok=True)
            config_path.unlink(missing_ok=True)

            free_gpu()
            started = time.time()
            model, history, best_point = train_model(cfg)

            ps = (
                evaluate(model, DATA["val1"])
                .sort_values("slate_id")
                .reset_index(drop=True)
            )
            minutes = (time.time() - started) / 60

            history.to_csv(run_dir / "history.csv", index=False)
            ps.to_parquet(metrics_path, index=False)

            torch.save(
                {
                    "state_dict": {
                        k: v.detach().cpu()
                        for k, v in model.state_dict().items()
                    },
                    "cfg": cfg,
                    "scalar_names": scalar_names,
                    "scaler": SCALER,
                    "best_check": best_point,
                    "metric": "purchase_ndcg@10",
                    "vocab": VOCAB,
                },
                run_dir / "checkpoint.pt",
            )
            config_path.write_text(json.dumps(cfg, indent=2))

            del model, history
            free_gpu()

        assert ps["slate_id"].is_unique
        assert np.isfinite(ps[METRICS].to_numpy()).all()

        if reference is None:
            reference = ps.copy()

        assert ps["slate_id"].equals(reference["slate_id"]), (
            f"{tag}: validation slates differ from full"
        )

        row = dict(
            variant=name,
            seed=seed,
            best_check=best_point,
            n_scalars=len(cols),
            n_eval_slates=len(ps),
            minutes=minutes,
            **{
                f"val1_{metric}": float(ps[metric].mean())
                for metric in METRICS
            },
        )
        row["delta_vs_full"] = float(
            (ps["ndcg@10"] - reference["ndcg@10"]).mean()
        )

        # Per-run completion record, including feature names.
        result_path.write_text(
            json.dumps({**row, "scalar_names": scalar_names}, indent=2)
        )

        rows.append(row)
        previous_rows[(name, seed)] = row
        manifest[tag] = scalar_names

        # Preserve other saved runs while updating the current one.
        pd.DataFrame(previous_rows.values()).to_csv(runs_path, index=False)
        manifest_path.write_text(json.dumps(manifest, indent=2))

        print(row)
        del ps



full_seed0: 45 scalar inputs
No matching completed run found; training.
epoch 0.50: train loss 0.4204 | val1 NDCG@10 0.3614 | val1 Hit@10 0.6015 | 0.9 min
epoch 1.00: train loss 0.4060 | val1 NDCG@10 0.3651 | val1 Hit@10 0.6067 | 0.9 min
epoch 1.50: train loss 0.3225 | val1 NDCG@10 0.3143 | val1 Hit@10 0.5409 | 0.9 min
epoch 2.00: train loss 0.3296 | val1 NDCG@10 0.3176 | val1 Hit@10 0.5462 | 0.9 min
epoch 2.50: train loss 0.2340 | val1 NDCG@10 0.2640 | val1 Hit@10 0.4769 | 0.9 min
{'variant': 'full', 'seed': 0, 'best_check': 2, 'n_scalars': 45, 'n_eval_slates': 1945, 'minutes': 4.745620969931284, 'val1_ndcg@10': 0.36506639843828514, 'val1_hit@10': 0.6066838046272494, 'val1_hit@1': 0.18118251928020565, 'val1_mrr': 0.3121097550635632, 'delta_vs_full': 0.0}

no_click_seed0: 33 scalar inputs
No matching completed run found; training.
epoch 0.50: train loss 0.4239 | val1 NDCG@10 0.3526 | val1 Hit@10 0.5977 | 0.8 min
epoch 1.00: train loss 0.4080 | val1 NDCG@10 0.3548 | val1 Hit@10 0.5990 

In [35]:
runs = pd.DataFrame(rows)
summary = runs.groupby("variant", sort=False).agg(
    val1_ndcg10_mean=("val1_ndcg@10", "mean"),
    val1_ndcg10_std=("val1_ndcg@10", "std"),
    delta_vs_full_mean=("delta_vs_full", "mean"),
    n_seeds=("seed", "nunique"),
    minutes_total=("minutes", "sum"),
).sort_values("val1_ndcg10_mean", ascending=False)
summary.to_csv(OUT_DIR / "ablation_summary.csv")
display(summary.round(6))
winner = str(summary.index[0])
selection = {"variant": winner, "overrides": ABLATIONS[winner],
             "seeds": SEEDS, "selection_metric": "val1 purchase NDCG@10",
             "val1_ndcg10_mean": float(summary.iloc[0].val1_ndcg10_mean)}
(OUT_DIR / "selected_config.json").write_text(json.dumps(selection, indent=2))
print("Selected configuration:", selection)
print("A one-seed standard deviation is undefined (NaN). Val2/test have not been loaded.")


,val1_ndcg10_mean,val1_ndcg10_std,delta_vs_full_mean,n_seeds,minutes_total
variant,,,,,
no_dominant_brands,0.366100,0.001929,0.001514,3,14.246625
no_candidate_item_ID,0.364979,0.000829,0.000394,3,13.582313
full,0.364586,0.001084,0.000000,3,14.248124
no_attention,0.364285,0.001292,-0.000301,3,12.608435
no_recency_decay,0.364159,0.001169,-0.000427,3,14.249583
no_history_query,0.363668,0.001467,-0.000918,3,13.391288
no_candidate_name,0.363181,0.001621,-0.001405,3,13.907107
no_cart,0.360470,0.001316,-0.004115,3,11.411839
no_order,0.359463,0.002244,-0.005123,3,12.116733


Selected configuration: {'variant': 'no_dominant_brands', 'overrides': {'use_dominant_brand_ids': False}, 'seeds': [0, 1, 2], 'selection_metric': 'val1 purchase NDCG@10', 'val1_ndcg10_mean': 0.36609968479063104}
A one-seed standard deviation is undefined (NaN). Val2/test have not been loaded.


In [36]:
def load_seed_metrics(variant):
    """Load aligned per-search metrics: arrays have shape [seeds, searches]."""
    frames = [
        pd.read_parquet(
            OUT_DIR / f"{variant}_seed{s}" / "val1_per_slate.parquet"
        ).sort_values("slate_id").reset_index(drop=True)
        for s in SEEDS
    ]

    slate_ids = frames[0]["slate_id"]
    assert slate_ids.is_unique

    for frame in frames:
        assert frame["slate_id"].is_unique
        assert frame["slate_id"].equals(slate_ids), "Slates differ across seeds"

    return (
        slate_ids,
        np.stack([f["ndcg@10"].to_numpy() for f in frames]),
        np.stack([f["hit@10"].to_numpy() for f in frames]),
    )


def bootstrap_mean_diff(d, n_boot=2000, seed=0):
    """Bootstrap searches after averaging paired differences across seeds."""
    d = np.asarray(d, dtype=float)

    if not len(d) or not np.isfinite(d).all():
        raise ValueError("Expected nonempty, finite differences")

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    batch_size = max(1, min(100, 1_000_000 // len(d)))

    for start in range(0, n_boot, batch_size):
        size = min(batch_size, n_boot - start)
        indices = rng.integers(0, len(d), size=(size, len(d)))
        boot[start:start + size] = d[indices].mean(axis=1)

    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi)


full_ids, full_ndcg, _ = load_seed_metrics("full")
run_info = pd.read_csv(OUT_DIR / "ablation_runs.csv")

rows = []

for variant in ABLATIONS:
    slate_ids, ndcg, hit = load_seed_metrics(variant)
    assert slate_ids.equals(full_ids), "Slates differ from full"

    # One paired difference per search, averaged over training seeds.
    mean_diff_per_slate = (ndcg - full_ndcg).mean(axis=0)
    diff, lo, hi = bootstrap_mean_diff(mean_diff_per_slate)

    counts = run_info.loc[
        (run_info["variant"] == variant)
        & run_info["seed"].isin(SEEDS),
        "n_scalars",
    ].unique()
    assert len(counts) == 1, "Scalar count differs across seeds or is missing"

    rows.append({
        "configuration": variant,
        "n_scalars": int(counts[0]),
        "seeds": len(SEEDS),
        "ndcg@10": ndcg.mean(),
        "seed_sd": ndcg.mean(axis=1).std(ddof=1),
        "hit@10": hit.mean(),
        "diff": diff,
        "ci_low": lo,
        "ci_high": hi,
        "ci_excludes_0": "*" if lo > 0 or hi < 0 else "",
    })

comparison = pd.DataFrame(rows).set_index("configuration")
comparison.to_csv(OUT_DIR / "ablation_average_bootstrap.csv")

display(comparison.round(4))

,n_scalars,seeds,ndcg@10,seed_sd,hit@10,diff,ci_low,ci_high,ci_excludes_0
configuration,,,,,,,,,
full,45,3,0.3646,0.0011,0.6062,0.0000,0.0000,0.0000,
no_click,33,3,0.3566,0.0018,0.6016,-0.0080,-0.0125,-0.0033,*
no_cart,33,3,0.3605,0.0013,0.6022,-0.0041,-0.0078,-0.0008,*
no_order,33,3,0.3595,0.0022,0.6038,-0.0051,-0.0084,-0.0019,*
no_history_query,45,3,0.3637,0.0015,0.6047,-0.0009,-0.0022,0.0003,
no_candidate_name,45,3,0.3632,0.0016,0.6112,-0.0014,-0.0046,0.0018,
no_attributes,45,3,0.3543,0.0015,0.5910,-0.0103,-0.0151,-0.0054,*
no_recency_decay,45,3,0.3642,0.0012,0.6062,-0.0004,-0.0015,0.0005,
no_step2c_scalars,17,3,0.3547,0.0009,0.5954,-0.0099,-0.0141,-0.0057,*


## Interpretation
A negative `delta_vs_full` means removing that component lowered val1 NDCG@10. Compare across seeds before attributing small changes to a component. All checkpoints and configurations were selected using val1, so these scores are selection results. Freeze the chosen configuration before subsequent val2 calibration or final test evaluation; neither is performed here.
